In [1]:
# ============================================================
# Stage 03: Traditional NLP Model
# Step 3.1: Setup & Configuration
# ============================================================

from pathlib import Path

import pandas as pd
import sklearn

# Automatically locate project directory
CURRENT_DIR = Path.cwd().resolve()

PROJECT_DIR = next(
    (
        path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data" / "processed" / "train.csv").exists()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project directory or train.csv not found!"
    )

DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("=" * 55)
print("STAGE 03 - TRADITIONAL NLP MODEL")
print("=" * 55)

print("Project:", PROJECT_DIR)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Random State:", RANDOM_STATE)


STAGE 03 - TRADITIONAL NLP MODEL
Project: C:\Users\faisa\Desktop\NLP Project
Pandas: 2.3.3
Scikit-learn: 1.8.0
Random State: 42


In [2]:
# ============================================================
# Step 3.1 - Load Training & Development Datasets
# ============================================================

# Load data
train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype=str,
    keep_default_na=False
)

dev_df = pd.read_csv(
    DATA_DIR / "dev.csv",
    dtype=str,
    keep_default_na=False
)

# Expected dataset structure
required_columns = {"CAMPNO", "text", "label"}

assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(dev_df.columns)

# Verify expected dataset sizes
assert len(train_df) == 3558
assert len(dev_df) == 762

# Separate inputs and labels
X_train = train_df["text"]
y_train = train_df["label"]

X_dev = dev_df["text"]
y_dev = dev_df["label"]

# Verify valid input
assert X_train.str.strip().ne("").all()
assert X_dev.str.strip().ne("").all()

# Verify same target classes
assert set(y_train.unique()) == set(y_dev.unique())
assert y_train.nunique() == 5

# Verify no exact overlap
assert set(train_df["CAMPNO"]).isdisjoint(
    set(dev_df["CAMPNO"])
)

assert set(X_train.str.casefold()).isdisjoint(
    set(X_dev.str.casefold())
)

# Display results
print("=" * 55)
print("DATA LOADING RESULTS")
print("=" * 55)

print("Training Examples:", len(X_train))
print("Development Examples:", len(X_dev))
print("Number of Classes:", y_train.nunique())

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nDevelopment Class Distribution:")
print(y_dev.value_counts())

print("\nSample Training Description:")
print(X_train.iloc[0][:500])

print("\nSample Label:")
print(y_train.iloc[0])

print("\nData loading verification: PASSED!")


DATA LOADING RESULTS
Training Examples: 3558
Development Examples: 762
Number of Classes: 5

Training Class Distribution:
label
Electrical System    1323
Service Brakes        685
Air Bags              552
Fuel System           516
Steering              482
Name: count, dtype: int64

Development Class Distribution:
label
Electrical System    283
Service Brakes       147
Air Bags             118
Fuel System          110
Steering             104
Name: count, dtype: int64

Sample Training Description:
Collins is recalling certain model year 2012 Collins, Mid-Bus, and Corbeil SL, SH, DH, and DE model school and multi-function school activity buses manufactured from March 12, 2012, through July 2, 2012. The electrical panel cover above the driver may not be adequately secured. Vibration can cause the designed disks to lose grip and release the cover without warning.

Sample Label:
Electrical System

Data loading verification: PASSED!


In [3]:
# ============================================================
# Step 3.2 - TF-IDF Vectorization
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# 2. Learn vocabulary from Training data only
X_train_tfidf = tfidf.fit_transform(X_train)

# 3. Transform Development data using same vocabulary
X_dev_tfidf = tfidf.transform(X_dev)

# ============================================================
# Verification
# ============================================================

print("=" * 55)
print("TF-IDF VECTORIZATION RESULTS")
print("=" * 55)

print("Training Matrix Shape:", X_train_tfidf.shape)
print("Development Matrix Shape:", X_dev_tfidf.shape)

print("\nVocabulary Size:", len(tfidf.vocabulary_))

print("Training Matrix Type:", type(X_train_tfidf).__name__)

print("\nFirst 20 Features:")
print(tfidf.get_feature_names_out()[:20])

# Verify dimensions
assert X_train_tfidf.shape[0] == len(X_train)
assert X_dev_tfidf.shape[0] == len(X_dev)

assert X_train_tfidf.shape[1] == X_dev_tfidf.shape[1]

print("\nTF-IDF Vectorization: PASSED!")


TF-IDF VECTORIZATION RESULTS
Training Matrix Shape: (3558, 20146)
Development Matrix Shape: (762, 20146)

Vocabulary Size: 20146
Training Matrix Type: csr_matrix

First 20 Features:
['000' '000 lbs' '01' '01 2011' '01 4012506' '01 4013126' '01 4013235'
 '02' '02 4012909' '02 4013126' '02 4013235' '04' '04091' '04091 dual'
 '04azs' '04ewm' '0504501' '0504501 0504504' '0504504' '0504504 or']

TF-IDF Vectorization: PASSED!


In [4]:
# ============================================================
# Step 3.2 - Inspect TF-IDF Features
# ============================================================

# Select first training example
sample_text = X_train.iloc[0]

# Get its TF-IDF vector
sample_vector = X_train_tfidf[0]

# Feature names
feature_names = tfidf.get_feature_names_out()

# Extract non-zero features and their weights
features = pd.DataFrame({
    "Feature": feature_names[sample_vector.indices],
    "TF-IDF Weight": sample_vector.data
})

# Sort by highest weights
features = features.sort_values(
    "TF-IDF Weight",
    ascending=False
)

print("=" * 55)
print("TF-IDF FEATURE INSPECTION")
print("=" * 55)

print("\nOriginal Text:")
print(sample_text[:500])

print("\nTotal Active Features:", sample_vector.nnz)

print("\nTop 15 TF-IDF Features:")
display(features.head(15))


TF-IDF FEATURE INSPECTION

Original Text:
Collins is recalling certain model year 2012 Collins, Mid-Bus, and Corbeil SL, SH, DH, and DE model school and multi-function school activity buses manufactured from March 12, 2012, through July 2, 2012. The electrical panel cover above the driver may not be adequately secured. Vibration can cause the designed disks to lose grip and release the cover without warning.

Total Active Features: 87

Top 15 TF-IDF Features:


,Feature,TF-IDF Weight
0,collins,0.220661
25,cover,0.151823
9,sh,0.139123
51,dh and,0.139123
50,sh dh,0.139123
49,sl sh,0.139123
7,corbeil,0.139123
55,and multi,0.139123
56,multi function,0.139123
57,function school,0.139123


In [5]:
# ============================================================
# Step 3.3 - Logistic Regression Training
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# 1. Verify TF-IDF matrices
print("Training Matrix:", X_train_tfidf.shape)
print("Development Matrix:", X_dev_tfidf.shape)
print("Vocabulary Size:", len(tfidf.vocabulary_))

# 2. Initialize Logistic Regression
baseline_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# 3. Train on Training Set only
baseline_model.fit(
    X_train_tfidf,
    y_train
)

print("\nModel training completed!")

# 4. Predict Development Set
dev_predictions = baseline_model.predict(
    X_dev_tfidf
)

# 5. Calculate metrics
dev_accuracy = accuracy_score(
    y_dev,
    dev_predictions
)

dev_macro_f1 = f1_score(
    y_dev,
    dev_predictions,
    average="macro"
)

# 6. Display results
print("\n" + "=" * 55)
print("INITIAL DEVELOPMENT RESULTS")
print("=" * 55)

print(f"Accuracy: {dev_accuracy:.4f}")
print(f"Macro F1: {dev_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_dev,
        dev_predictions,
        digits=4,
        zero_division=0
    )
)

# 7. Verification
assert len(dev_predictions) == len(y_dev)
assert len(baseline_model.classes_) == 5

print("\nBaseline training verification: PASSED!")


Training Matrix: (3558, 20146)
Development Matrix: (762, 20146)
Vocabulary Size: 20146

Model training completed!

INITIAL DEVELOPMENT RESULTS
Accuracy: 0.9580
Macro F1: 0.9598

Classification Report:
                   precision    recall  f1-score   support

         Air Bags     0.9823    0.9407    0.9610       118
Electrical System     0.9121    0.9894    0.9492       283
      Fuel System     0.9810    0.9364    0.9581       110
   Service Brakes     1.0000    0.9524    0.9756       147
         Steering     0.9897    0.9231    0.9552       104

         accuracy                         0.9580       762
        macro avg     0.9730    0.9484    0.9598       762
     weighted avg     0.9604    0.9580    0.9582       762


Baseline training verification: PASSED!


In [6]:
# ============================================================
# Step 3.4 - Logistic Regression Hyperparameter Tuning
# ============================================================

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Hyperparameters
C_VALUES = [0.1, 1.0, 3.0, 10.0]
CLASS_WEIGHTS = [None, "balanced"]

results = []
trained_models = {}

print("=" * 55)
print("HYPERPARAMETER TUNING")
print("=" * 55)

# Train and evaluate each configuration
for c in C_VALUES:
    for weight in CLASS_WEIGHTS:

        model = LogisticRegression(
            C=c,
            class_weight=weight,
            max_iter=2000,
            random_state=42
        )

        # Train on Training Set only
        model.fit(X_train_tfidf, y_train)

        # Predict Development Set
        predictions = model.predict(X_dev_tfidf)

        # Calculate metrics
        accuracy = accuracy_score(y_dev, predictions)

        macro_f1 = f1_score(
            y_dev,
            predictions,
            average="macro"
        )

        # Store results
        results.append({
            "C": c,
            "class_weight": str(weight),
            "accuracy": accuracy,
            "macro_f1": macro_f1
        })

        trained_models[(c, str(weight))] = model

        print(
            f"C={c:<5} | "
            f"Weight={str(weight):<8} | "
            f"Accuracy={accuracy:.4f} | "
            f"Macro F1={macro_f1:.4f}"
        )

# ============================================================
# Compare all configurations
# ============================================================

tuning_results = pd.DataFrame(results)

tuning_results = tuning_results.sort_values(
    by=["macro_f1", "accuracy"],
    ascending=[False, False],
    kind="stable"
).reset_index(drop=True)

print("\n" + "=" * 55)
print("TUNING RESULTS - RANKED")
print("=" * 55)

display(tuning_results)

# ============================================================
# Select best model
# ============================================================

best_config = tuning_results.iloc[0]

best_c = float(best_config["C"])
best_weight = best_config["class_weight"]

best_traditional_model = trained_models[
    (best_c, best_weight)
]

print("\n" + "=" * 55)
print("BEST TRADITIONAL MODEL")
print("=" * 55)

print("Best C:", best_c)
print("Best Class Weight:", best_weight)

print(
    "Best Development Accuracy:",
    round(best_config["accuracy"], 4)
)

print(
    "Best Development Macro F1:",
    round(best_config["macro_f1"], 4)
)

print("\nHyperparameter tuning completed!")


HYPERPARAMETER TUNING
C=0.1   | Weight=None     | Accuracy=0.7585 | Macro F1=0.7458
C=0.1   | Weight=balanced | Accuracy=0.9554 | Macro F1=0.9570
C=1.0   | Weight=None     | Accuracy=0.9580 | Macro F1=0.9598
C=1.0   | Weight=balanced | Accuracy=0.9711 | Macro F1=0.9719
C=3.0   | Weight=None     | Accuracy=0.9633 | Macro F1=0.9648
C=3.0   | Weight=balanced | Accuracy=0.9738 | Macro F1=0.9740
C=10.0  | Weight=None     | Accuracy=0.9724 | Macro F1=0.9736
C=10.0  | Weight=balanced | Accuracy=0.9790 | Macro F1=0.9788

TUNING RESULTS - RANKED


,C,class_weight,accuracy,macro_f1
0,10.0,balanced,0.979003,0.978801
1,3.0,balanced,0.973753,0.973996
2,10.0,None,0.972441,0.973614
3,1.0,balanced,0.971129,0.971928
4,3.0,None,0.963255,0.964763
5,1.0,None,0.958005,0.959833
6,0.1,balanced,0.955381,0.956954
7,0.1,None,0.758530,0.745798



BEST TRADITIONAL MODEL
Best C: 10.0
Best Class Weight: balanced
Best Development Accuracy: 0.979
Best Development Macro F1: 0.9788

Hyperparameter tuning completed!


In [7]:
# ============================================================
# Step 3.5 - Save & Verify Traditional NLP Model
# ============================================================

import joblib
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

# 1. Combine fitted TF-IDF and trained classifier
traditional_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", best_traditional_model)
])

# 2. Verify predictions before saving
predictions_before = traditional_pipeline.predict(X_dev)

accuracy_before = accuracy_score(
    y_dev, predictions_before
)

f1_before = f1_score(
    y_dev,
    predictions_before,
    average="macro"
)

# 3. Define model path
MODEL_PATH = MODEL_DIR / "traditional_tfidf_logreg.joblib"

# 4. Save complete pipeline
joblib.dump(traditional_pipeline, MODEL_PATH)

print("Model saved:", MODEL_PATH.name)

# 5. Load saved pipeline
loaded_pipeline = joblib.load(MODEL_PATH)

# 6. Predict again after loading
predictions_after = loaded_pipeline.predict(X_dev)

# 7. Verify identical predictions
assert np.array_equal(
    predictions_before,
    predictions_after
), "Saved model predictions do not match!"

# 8. Verify selected model performance
assert np.isclose(
    accuracy_before,
    best_config["accuracy"]
)

assert np.isclose(
    f1_before,
    best_config["macro_f1"]
)

# ============================================================
# Results
# ============================================================

print("\n" + "=" * 55)
print("MODEL SAVING VERIFICATION")
print("=" * 55)

print("Model:", MODEL_PATH.name)
print(
    "File Size (MB):",
    round(MODEL_PATH.stat().st_size / 1024**2, 2)
)

print(f"Development Accuracy: {accuracy_before:.4f}")
print(f"Development Macro F1: {f1_before:.4f}")

print("\nPredictions Before Saving:", len(predictions_before))
print("Predictions After Loading:", len(predictions_after))

print("\nSaved Model Integrity: PASSED!")


Model saved: traditional_tfidf_logreg.joblib

MODEL SAVING VERIFICATION
Model: traditional_tfidf_logreg.joblib
File Size (MB): 1.24
Development Accuracy: 0.9790
Development Macro F1: 0.9788

Predictions Before Saving: 762
Predictions After Loading: 762

Saved Model Integrity: PASSED!
